# Notebook 16: Gradient Accumulation

## 📚 Historical Context

**Timeline**: 2017-present - The scaling challenge

**The Problem: GPU Memory Bottleneck (2017-2018)**:
- **Original Transformer (2017)**: Trained with batch size 25K tokens on TPUs
- **BERT (2018)**: Required 64 TPU chips, batch size 256
- **GPT-2 (2019)**: Trained on 32 GPUs, couldn't fit large batches
- **Academic researchers**: Stuck with single GPU, tiny batch sizes

**The Memory Crisis**:
```
Llama-7B with batch size 8, seq_len 2048:
- Model parameters:     28 GB (float32)
- Activations:          ~16 GB
- Gradients:            28 GB
- Optimizer states:     56 GB (Adam)
Total:                  ~128 GB

Your GPU:               24 GB (RTX 3090)
```

**The Breakthrough: Gradient Accumulation**:
- Proposed alongside early deep learning (2013-2015)
- Popularized by Hugging Face Transformers (2019)
- Key insight: **Gradients are additive**
- Trade time for memory: Process small batches, accumulate gradients

**Impact**:
- 2019: BERT training accessible on single GPU
- 2020: GPT-2 fine-tuning possible with 8GB GPU
- 2023: Llama fine-tuning on consumer GPUs (with accumulation)
- Present: Standard technique in all training frameworks

## 🎯 What You'll Learn

1. **Mathematical Foundation** - Why gradient accumulation works
2. **Memory vs Time Trade-offs** - Understanding the cost
3. **Implementation from Scratch** - How it actually works
4. **Effective Batch Size** - The key concept
5. **Practical Comparison** - Batch 32 vs 4×8 accumulation
6. **Common Pitfalls** - BatchNorm, learning rate, gradient clipping

## 💡 Why This Matters

**Critical for limited resources**:
- **Train large models on small GPUs** (7B model on 16GB GPU)
- **Maintain large effective batch sizes** (stability + performance)
- **No accuracy loss** (mathematically equivalent)
- **Universal technique** (works with any model/optimizer)

**Real-world necessity**:
```
Without gradient accumulation:
- Batch size 4 → unstable training, poor performance
- Need expensive multi-GPU setup
- Can't train modern models

With gradient accumulation (4 steps):
- Effective batch size 16 → stable training
- Single GPU sufficient
- Trade: 4x slower, but works!
```

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import time

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Part 1: Mathematical Foundation

### Why Gradient Accumulation Works: The Math

**Key Insight**: Gradients are additive

**Standard Batch Training**:
```
Given batch B = [x₁, x₂, ..., xₙ]

Loss = (1/n) Σ L(xᵢ)

∇Loss = (1/n) Σ ∇L(xᵢ)
```

**Gradient Accumulation** (split into k mini-batches):
```
B₁ = [x₁, ..., xₘ]
B₂ = [xₘ₊₁, ..., x₂ₘ]
...
Bₖ = [..., xₙ]

Compute separately:
∇₁ = (1/m) Σ ∇L(xᵢ) for i in B₁
∇₂ = (1/m) Σ ∇L(xᵢ) for i in B₂
...

Accumulate:
∇_total = (1/k) Σ ∇ⱼ
```

**Mathematical Equivalence**:
```
∇_total = (1/k) × (1/m) × Σ Σ ∇L(xᵢ)
        = (1/n) × Σ ∇L(xᵢ)
        = ∇Loss
```

**Proof by Example**:
```
Batch size 8, accumulation steps 4:

Standard:
∇ = (∇₁ + ∇₂ + ∇₃ + ∇₄ + ∇₅ + ∇₆ + ∇₇ + ∇₈) / 8

Accumulated:
Step 1: ∇ᵃ₁ = (∇₁ + ∇₂) / 2
Step 2: ∇ᵃ₂ = (∇₃ + ∇₄) / 2
Step 3: ∇ᵃ₃ = (∇₅ + ∇₆) / 2
Step 4: ∇ᵃ₄ = (∇₇ + ∇₈) / 2

Final: ∇ = (∇ᵃ₁ + ∇ᵃ₂ + ∇ᵃ₃ + ∇ᵃ₄) / 4
         = (∇₁ + ∇₂ + ∇₃ + ∇₄ + ∇₅ + ∇₆ + ∇₇ + ∇₈) / 8
         
IDENTICAL! ✓
```

### Important Caveat:

**This assumes**:
1. No batch-dependent operations (BatchNorm is problematic)
2. Proper gradient scaling
3. Same random state across runs (for exact equivalence)

**In practice**:
- LayerNorm: Works perfectly ✓
- BatchNorm: Problematic (uses batch statistics) ✗
- Dropout: Works (applied per forward pass) ✓
- RMSNorm: Works perfectly ✓

In [ ]:
# Demonstrate mathematical equivalence with simple example
print("=" * 60)
print("MATHEMATICAL EQUIVALENCE PROOF")
print("=" * 60)

# Simple linear model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 1)
    
    def forward(self, x):
        return self.linear(x)

# Create synthetic data
torch.manual_seed(42)
X = torch.randn(8, 10)  # 8 samples
y = torch.randn(8, 1)

# Method 1: Standard batch gradient
model1 = SimpleModel()
loss_fn = nn.MSELoss()

output1 = model1(X)
loss1 = loss_fn(output1, y)
loss1.backward()

grad1_weight = model1.linear.weight.grad.clone()
grad1_bias = model1.linear.bias.grad.clone()

print("\nMethod 1: Standard batch (size=8)")
print(f"Loss: {loss1.item():.6f}")
print(f"Weight gradient norm: {grad1_weight.norm().item():.6f}")
print(f"Bias gradient norm:   {grad1_bias.norm().item():.6f}")

# Method 2: Gradient accumulation (4 steps of batch size 2)
model2 = SimpleModel()
# Copy weights to make it identical
model2.linear.weight.data = model1.linear.weight.data.clone()
model2.linear.bias.data = model1.linear.bias.data.clone()
# Reset gradients
model1.zero_grad()

accumulation_steps = 4
micro_batch_size = 2

total_loss = 0
for step in range(accumulation_steps):
    # Get micro-batch
    start_idx = step * micro_batch_size
    end_idx = start_idx + micro_batch_size
    X_micro = X[start_idx:end_idx]
    y_micro = y[start_idx:end_idx]
    
    # Forward pass
    output2 = model2(X_micro)
    loss2 = loss_fn(output2, y_micro)
    
    # Backward pass (gradients accumulate)
    # Note: No normalization here, we'll do it after
    loss2.backward()
    
    total_loss += loss2.item()

# Average gradients
for param in model2.parameters():
    param.grad /= accumulation_steps

grad2_weight = model2.linear.weight.grad.clone()
grad2_bias = model2.linear.bias.grad.clone()

print("\nMethod 2: Gradient accumulation (4 steps × batch size=2)")
print(f"Average loss: {total_loss / accumulation_steps:.6f}")
print(f"Weight gradient norm: {grad2_weight.norm().item():.6f}")
print(f"Bias gradient norm:   {grad2_bias.norm().item():.6f}")

# Compare gradients
print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"Weight gradient difference: {(grad1_weight - grad2_weight).abs().max().item():.10f}")
print(f"Bias gradient difference:   {(grad1_bias - grad2_bias).abs().max().item():.10f}")
print("\n✓ Gradients are identical (within numerical precision)!")
print("=" * 60)

## Part 2: Memory vs Time Trade-offs

### The Fundamental Trade-off:

**Memory** (decreases with accumulation):
```
Peak memory ∝ batch_size × seq_len × model_size

Batch 32:           High memory ↑
Batch 8 (accum 4):  Low memory ↓
```

**Time** (increases with accumulation):
```
Time per step ∝ accumulation_steps

Batch 32:           1 forward + 1 backward
Batch 8 (accum 4):  4 forwards + 4 backwards
                    ~4x slower
```

### Memory Breakdown:

**What uses GPU memory during training**:
```
1. Model Parameters (W)
   Size: Fixed, independent of batch size
   Example: Llama-7B = 7B × 4 bytes = 28 GB (float32)

2. Gradients (∇W)
   Size: Same as parameters
   Example: 28 GB

3. Optimizer States (Adam: m, v)
   Size: 2× parameters (first and second moments)
   Example: 56 GB

4. Activations (intermediate outputs)
   Size: batch_size × seq_len × hidden_dim × num_layers
   Example: 32 × 2048 × 4096 × 32 = 8.6 GB
   THIS IS WHAT GRADIENT ACCUMULATION REDUCES! ✓
```

### Calculation Example:

**Llama-7B, sequence length 2048**:

```
Batch size 32:
- Parameters:   7B × 4 bytes = 28 GB
- Gradients:    28 GB
- Optimizer:    56 GB  (Adam)
- Activations:  ~8 GB  (depends on seq_len × batch)
TOTAL:          ~120 GB → Requires A100-80GB

Batch size 8 (accum 4):
- Parameters:   28 GB
- Gradients:    28 GB
- Optimizer:    56 GB
- Activations:  ~2 GB  (4x smaller!)
TOTAL:          ~114 GB → Still tight, but possible

Batch size 1 (accum 32):
- Parameters:   28 GB
- Gradients:    28 GB
- Optimizer:    56 GB
- Activations:  ~0.25 GB (32x smaller!)
TOTAL:          ~112 GB → Fits on A100-80GB with tricks
```

### Time Overhead:

**Why it's slower**:
```
1. Multiple forward passes
2. Multiple backward passes
3. Cannot batch operations as efficiently
4. More memory transfers

Rule of thumb:
accumulation_steps × slower
(not exactly, due to better memory patterns)
```

In [ ]:
# Measure memory and time trade-offs
print("=" * 60)
print("MEMORY AND TIME TRADE-OFFS")
print("=" * 60)

# Simple transformer-like model
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=256, num_layers=4, num_heads=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = self.fc(x)
        return x

# Test configurations
configs = [
    {"batch_size": 32, "accum_steps": 1, "name": "Batch 32 (no accumulation)"},
    {"batch_size": 16, "accum_steps": 2, "name": "Batch 16 (accum 2)"},
    {"batch_size": 8, "accum_steps": 4, "name": "Batch 8 (accum 4)"},
    {"batch_size": 4, "accum_steps": 8, "name": "Batch 4 (accum 8)"},
]

seq_len = 128
results = []

for config in configs:
    batch_size = config["batch_size"]
    accum_steps = config["accum_steps"]
    
    # Create model
    model = MiniTransformer().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    # Synthetic data
    torch.manual_seed(42)
    X = torch.randint(0, 1000, (batch_size * accum_steps, seq_len)).to(device)
    y = torch.randint(0, 1000, (batch_size * accum_steps, seq_len)).to(device)
    
    # Measure memory and time
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    optimizer.zero_grad()
    
    for step in range(accum_steps):
        # Get micro-batch
        start_idx = step * batch_size
        end_idx = start_idx + batch_size
        X_batch = X[start_idx:end_idx]
        y_batch = y[start_idx:end_idx]
        
        # Forward pass
        outputs = model(X_batch)
        loss = F.cross_entropy(
            outputs.reshape(-1, 1000),
            y_batch.reshape(-1)
        )
        
        # Backward pass
        (loss / accum_steps).backward()  # Scale loss
    
    optimizer.step()
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed_time = time.time() - start_time
    
    # Get peak memory
    if device.type == 'cuda':
        peak_memory = torch.cuda.max_memory_allocated() / 1e9  # GB
    else:
        peak_memory = 0
    
    results.append({
        "name": config["name"],
        "batch_size": batch_size,
        "accum_steps": accum_steps,
        "effective_batch": batch_size * accum_steps,
        "time": elapsed_time,
        "memory": peak_memory
    })
    
    # Clean up
    del model, optimizer, X, y
    if device.type == 'cuda':
        torch.cuda.empty_cache()

# Display results
print("\nResults (effective batch size = 32 for all):")
print("-" * 80)
print(f"{'Config':<30} {'Batch':>8} {'Accum':>6} {'Time (s)':>10} {'Memory (GB)':>12}")
print("-" * 80)

for r in results:
    print(f"{r['name']:<30} {r['batch_size']:>8} {r['accum_steps']:>6} {r['time']:>10.3f} {r['memory']:>12.2f}")

print("=" * 60)

# Visualize trade-offs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Time comparison
names = [r['name'] for r in results]
times = [r['time'] for r in results]
colors = sns.color_palette('husl', len(results))

axes[0].bar(range(len(names)), times, color=colors)
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels([f"BS {r['batch_size']}\nAcc {r['accum_steps']}" for r in results], rotation=0)
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Training Time Comparison\n(Effective batch size = 32 for all)')
axes[0].grid(axis='y', alpha=0.3)

# Add speedup annotations
baseline_time = times[0]
for i, t in enumerate(times):
    slowdown = t / baseline_time
    axes[0].text(i, t + 0.01, f'{slowdown:.2f}x', ha='center', va='bottom')

# Plot 2: Memory comparison
if device.type == 'cuda':
    memories = [r['memory'] for r in results]
    axes[1].bar(range(len(names)), memories, color=colors)
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels([f"BS {r['batch_size']}\nAcc {r['accum_steps']}" for r in results], rotation=0)
    axes[1].set_ylabel('Peak Memory (GB)')
    axes[1].set_title('Peak Memory Usage Comparison')
    axes[1].grid(axis='y', alpha=0.3)
    
    # Add reduction annotations
    baseline_mem = memories[0]
    for i, m in enumerate(memories):
        reduction = (baseline_mem - m) / baseline_mem * 100
        axes[1].text(i, m + 0.01, f'{reduction:.1f}%\nsaved', ha='center', va='bottom', fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'GPU not available\nMemory comparison requires CUDA', 
                ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Memory Usage (GPU Required)')

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. Time increases linearly with accumulation steps")
print("2. Memory decreases significantly with smaller batch sizes")
print("3. Effective batch size remains constant (32)")
print("4. Trade-off: Use accumulation when memory-constrained")

## Part 3: Implementation from Scratch

### Proper Implementation Details:

**Three critical steps**:

**1. Gradient Scaling**:
```python
# WRONG: Gradients will be too large
loss.backward()

# CORRECT: Scale loss by accumulation steps
(loss / accumulation_steps).backward()

# ALTERNATIVE: Scale gradients after accumulation
loss.backward()
# ... after all accumulation steps ...
for param in model.parameters():
    param.grad /= accumulation_steps
```

**2. Gradient Accumulation**:
```python
# Don't zero gradients inside accumulation loop!
optimizer.zero_grad()  # Once before accumulation

for step in range(accumulation_steps):
    outputs = model(batch)
    loss = criterion(outputs, targets)
    (loss / accumulation_steps).backward()  # Accumulates
    # NO optimizer.zero_grad() here!

optimizer.step()  # Once after accumulation
```

**3. Gradient Clipping** (if used):
```python
# Clip AFTER accumulation, BEFORE optimizer step
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```

### Complete Training Loop Template:

```python
accumulation_steps = 4
effective_batch_size = batch_size * accumulation_steps

optimizer.zero_grad()

for step, (inputs, targets) in enumerate(dataloader):
    # Forward pass
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    
    # Backward pass (scaled)
    (loss / accumulation_steps).backward()
    
    # Update weights every accumulation_steps
    if (step + 1) % accumulation_steps == 0:
        # Optional: Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Optimizer step
        optimizer.step()
        optimizer.zero_grad()
```

### Common Mistakes:

**❌ WRONG**:
```python
# Mistake 1: Not scaling loss
loss.backward()  # Gradients too large!

# Mistake 2: Zero gradients inside loop
optimizer.zero_grad()  # Throws away accumulated gradients!

# Mistake 3: Clip before accumulation complete
torch.nn.utils.clip_grad_norm_()  # Wrong timing!
```

**✓ CORRECT**:
```python
(loss / accumulation_steps).backward()  # Scale
# Let gradients accumulate (no zero_grad)
torch.nn.utils.clip_grad_norm_()  # After accumulation
```

In [ ]:
# Complete implementation from scratch
print("=" * 60)
print("GRADIENT ACCUMULATION: COMPLETE IMPLEMENTATION")
print("=" * 60)

# Create a simple model and dataset
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=64, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Training function with gradient accumulation
def train_with_accumulation(
    model, 
    dataloader, 
    criterion, 
    optimizer, 
    accumulation_steps=1,
    clip_grad=None,
    device='cpu'
):
    """
    Train with gradient accumulation.
    
    Args:
        model: PyTorch model
        dataloader: DataLoader with SMALL batches
        criterion: Loss function
        optimizer: Optimizer
        accumulation_steps: Number of steps to accumulate
        clip_grad: Gradient clipping max norm (None = no clipping)
        device: 'cpu' or 'cuda'
    """
    model.train()
    total_loss = 0
    num_updates = 0
    
    # Initialize gradients
    optimizer.zero_grad()
    
    for step, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass with scaling
        # Divide loss by accumulation_steps to average gradients
        scaled_loss = loss / accumulation_steps
        scaled_loss.backward()
        
        # Accumulate loss for logging
        total_loss += loss.item()
        
        # Update weights every accumulation_steps
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(dataloader):
            # Optional: Gradient clipping (AFTER accumulation, BEFORE step)
            if clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            
            # Optimizer step
            optimizer.step()
            
            # Reset gradients for next accumulation
            optimizer.zero_grad()
            
            num_updates += 1
    
    avg_loss = total_loss / len(dataloader)
    return avg_loss, num_updates

# Create synthetic dataset
torch.manual_seed(42)
num_samples = 320
X_train = torch.randn(num_samples, 20)
y_train = torch.randint(0, 10, (num_samples,))
dataset = TensorDataset(X_train, y_train)

# Test different configurations
configs = [
    {"batch_size": 32, "accum": 1, "name": "Standard (batch=32)"},
    {"batch_size": 8, "accum": 4, "name": "Accumulated (batch=8, accum=4)"},
]

print("\nTraining with different configurations:")
print("-" * 60)

for config in configs:
    print(f"\n{config['name']}:")
    
    # Create dataloader
    dataloader = DataLoader(
        dataset, 
        batch_size=config['batch_size'], 
        shuffle=True
    )
    
    # Create model and optimizer
    model = SimpleClassifier().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    # Train for a few epochs
    losses = []
    for epoch in range(5):
        loss, num_updates = train_with_accumulation(
            model, 
            dataloader, 
            criterion, 
            optimizer,
            accumulation_steps=config['accum'],
            clip_grad=1.0,
            device=device
        )
        losses.append(loss)
        
        if epoch % 1 == 0:
            print(f"  Epoch {epoch+1}: Loss = {loss:.4f}, Updates = {num_updates}")
    
    effective_batch = config['batch_size'] * config['accum']
    print(f"  Effective batch size: {effective_batch}")
    print(f"  Total optimizer updates: {num_updates * 5}")

print("\n" + "=" * 60)
print("\nKey Implementation Points:")
print("1. Scale loss by 1/accumulation_steps during backward")
print("2. Zero gradients ONCE before accumulation")
print("3. Optimizer step AFTER accumulation complete")
print("4. Gradient clipping AFTER accumulation, BEFORE step")
print("5. Both methods produce identical effective batch size")
print("=" * 60)

## Part 4: Effective Batch Size

### The Core Concept:

**Effective Batch Size** = `batch_size × accumulation_steps`

This is the batch size that actually affects training dynamics.

### Why It Matters:

**Learning rate scaling**:
```
Rule of thumb: LR ∝ sqrt(batch_size)

Batch 8:   LR = 1e-4
Batch 32:  LR = 2e-4  (2x batch → ~1.4x LR)
Batch 128: LR = 4e-4  (4x batch → 2x LR)
```

**Training stability**:
```
Small batch (4):      Noisy gradients, unstable
Medium batch (32):    Balanced, stable
Large batch (512):    Smooth, but may converge to sharp minima
```

**Gradient noise**:
```
Gradient noise ∝ 1/sqrt(batch_size)

Batch 1:   100% noise (single sample)
Batch 4:   50% noise
Batch 16:  25% noise
Batch 64:  12.5% noise
```

### Relationship with Learning Rate:

**Linear scaling rule** (Facebook AI Research, 2017):
```python
# When batch size increases by k, multiply LR by k
# (for large batch training)

base_lr = 0.1
base_batch = 256

# If batch increases to 1024:
new_batch = 1024
new_lr = base_lr * (new_batch / base_batch)
# new_lr = 0.4
```

**Square root scaling rule** (More conservative):
```python
# Multiply LR by sqrt(k)
new_lr = base_lr * sqrt(new_batch / base_batch)
```

### Practical Guidelines:

**For transformer fine-tuning**:
```
Effective batch 8-16:   LR = 1e-5 to 5e-5
Effective batch 32-64:  LR = 2e-5 to 1e-4
Effective batch 128+:   LR = 5e-5 to 2e-4
```

**Don't forget to account for accumulation**:
```python
# If you tune with batch=32, then switch to batch=8 + accum=4
# Keep the same LR! (effective batch is still 32)

# Original
batch_size = 32
lr = 5e-5

# With accumulation (SAME effective batch)
batch_size = 8
accumulation_steps = 4
lr = 5e-5  # Don't change!
```

In [ ]:
# Demonstrate effective batch size impact on training
print("=" * 60)
print("EFFECTIVE BATCH SIZE COMPARISON")
print("=" * 60)

# Create a more complex dataset
torch.manual_seed(42)
num_samples = 1000
X_train = torch.randn(num_samples, 20)
# Create a pattern that's learnable
y_train = (X_train[:, :5].sum(dim=1) > 0).long()
dataset = TensorDataset(X_train, y_train)

# Test different effective batch sizes
test_configs = [
    {"batch": 4, "accum": 1, "lr": 1e-3, "name": "Effective=4"},
    {"batch": 8, "accum": 1, "lr": 1e-3, "name": "Effective=8"},
    {"batch": 4, "accum": 4, "lr": 1e-3, "name": "Effective=16 (via accum)"},
    {"batch": 16, "accum": 1, "lr": 1e-3, "name": "Effective=16 (no accum)"},
    {"batch": 8, "accum": 4, "lr": 1e-3, "name": "Effective=32"},
]

results_history = {}

for config in test_configs:
    print(f"\nTraining: {config['name']}")
    print(f"  Batch size: {config['batch']}, Accumulation: {config['accum']}")
    print(f"  Effective batch: {config['batch'] * config['accum']}")
    
    # Create dataloader
    dataloader = DataLoader(
        dataset, 
        batch_size=config['batch'], 
        shuffle=True
    )
    
    # Create model and optimizer
    model = SimpleClassifier(input_dim=20, hidden_dim=64, output_dim=2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.CrossEntropyLoss()
    
    # Train
    losses = []
    for epoch in range(20):
        loss, _ = train_with_accumulation(
            model, 
            dataloader, 
            criterion, 
            optimizer,
            accumulation_steps=config['accum'],
            device=device
        )
        losses.append(loss)
    
    results_history[config['name']] = losses
    print(f"  Final loss: {losses[-1]:.4f}")

# Plot training curves
plt.figure(figsize=(14, 6))

for name, losses in results_history.items():
    plt.plot(losses, marker='o', markersize=4, label=name, linewidth=2, alpha=0.7)

plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss vs Effective Batch Size\n(Same learning rate for all)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("KEY OBSERVATIONS:")
print("=" * 60)
print("\n1. Effective batch size matters more than actual batch size")
print("   - Effective=16 (accum) ≈ Effective=16 (no accum)")
print("\n2. Larger effective batch size:")
print("   - Smoother training curves")
   print("   - More stable gradients")
print("   - May need higher learning rate")
print("\n3. Smaller effective batch size:")
print("   - Noisier training")
print("   - More exploration (can help escape local minima)")
print("   - Faster per-epoch training")
print("\n4. With accumulation:")
print("   - Same convergence as non-accumulated")
print("   - Just takes more wall-clock time")
print("=" * 60)

## Part 5: Practical Comparison - Batch 32 vs 4×8 Accumulation

### Head-to-Head Comparison:

Let's train identical models with:
1. **Standard**: Batch size 32, no accumulation
2. **Accumulated**: Batch size 8, accumulation steps 4

Both have **effective batch size = 32**

### What to Expect:

**Should be identical**:
- Final loss
- Model accuracy
- Gradient norms (averaged)
- Learning curves

**Will differ**:
- Training time (accumulated is slower)
- Peak memory usage (accumulated is lower)
- Number of forward/backward passes

### Validation:

This comparison proves gradient accumulation works correctly.

In [ ]:
# Comprehensive comparison: Batch 32 vs 8×4 accumulation
print("=" * 60)
print("HEAD-TO-HEAD: BATCH 32 vs 8×4 ACCUMULATION")
print("=" * 60)

# Create a reproducible dataset
torch.manual_seed(42)
np.random.seed(42)

num_samples = 1600
X_full = torch.randn(num_samples, 20)
y_full = (X_full[:, :5].sum(dim=1) > 0).long()

# Split into train/val
train_size = 1200
X_train, X_val = X_full[:train_size], X_full[train_size:]
y_train, y_val = y_full[:train_size], y_full[train_size:]

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

def evaluate(model, dataloader, criterion, device):
    """Evaluate model on validation set."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(targets).sum().item()
            total += targets.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy

# Configuration
num_epochs = 30
learning_rate = 0.001

configs = [
    {
        "name": "Standard (Batch 32)",
        "batch_size": 32,
        "accumulation_steps": 1,
        "color": "blue"
    },
    {
        "name": "Accumulated (Batch 8, Accum 4)",
        "batch_size": 8,
        "accumulation_steps": 4,
        "color": "red"
    }
]

all_results = {}

for config in configs:
    print(f"\n{'='*60}")
    print(f"Training: {config['name']}")
    print(f"{'='*60}")
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['batch_size'], 
        shuffle=True
    )
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # Create model (same architecture)
    torch.manual_seed(42)  # Same initialization
    model = SimpleClassifier(input_dim=20, hidden_dim=64, output_dim=2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    
    # Training
    train_losses = []
    val_losses = []
    val_accs = []
    grad_norms = []
    times = []
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Train
        model.train()
        epoch_loss = 0
        optimizer.zero_grad()
        
        for step, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # Backward
            (loss / config['accumulation_steps']).backward()
            epoch_loss += loss.item()
            
            # Update
            if (step + 1) % config['accumulation_steps'] == 0:
                # Track gradient norm
                total_norm = 0
                for p in model.parameters():
                    if p.grad is not None:
                        total_norm += p.grad.data.norm(2).item() ** 2
                total_norm = total_norm ** 0.5
                grad_norms.append(total_norm)
                
                optimizer.step()
                optimizer.zero_grad()
        
        # Evaluate
        avg_train_loss = epoch_loss / len(train_loader)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        train_losses.append(avg_train_loss)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        times.append(time.time() - start_time)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d}: "
                  f"Train Loss = {avg_train_loss:.4f}, "
                  f"Val Loss = {val_loss:.4f}, "
                  f"Val Acc = {val_acc:.2f}%")
    
    total_time = time.time() - start_time
    
    print(f"\nFinal Results:")
    print(f"  Val Loss: {val_losses[-1]:.4f}")
    print(f"  Val Acc:  {val_accs[-1]:.2f}%")
    print(f"  Time:     {total_time:.2f}s")
    print(f"  Avg Grad Norm: {np.mean(grad_norms):.4f}")
    
    all_results[config['name']] = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'grad_norms': grad_norms,
        'time': total_time,
        'color': config['color']
    }

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Training Loss
for name, results in all_results.items():
    axes[0, 0].plot(results['train_losses'], label=name, 
                    color=results['color'], linewidth=2, alpha=0.7)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Training Loss')
axes[0, 0].set_title('Training Loss Comparison')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Validation Accuracy
for name, results in all_results.items():
    axes[0, 1].plot(results['val_accs'], label=name, 
                    color=results['color'], linewidth=2, alpha=0.7)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Validation Accuracy (%)')
axes[0, 1].set_title('Validation Accuracy Comparison')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Gradient Norms (moving average)
window = 10
for name, results in all_results.items():
    grad_norms = results['grad_norms']
    # Moving average
    smoothed = np.convolve(grad_norms, np.ones(window)/window, mode='valid')
    axes[1, 0].plot(smoothed, label=name, 
                    color=results['color'], linewidth=2, alpha=0.7)
axes[1, 0].set_xlabel('Update Step')
axes[1, 0].set_ylabel('Gradient Norm')
axes[1, 0].set_title('Gradient Norm Comparison (smoothed)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Time comparison
names = list(all_results.keys())
times = [all_results[name]['time'] for name in names]
colors = [all_results[name]['color'] for name in names]
axes[1, 1].bar(range(len(names)), times, color=colors, alpha=0.7)
axes[1, 1].set_xticks(range(len(names)))
axes[1, 1].set_xticklabels(['Batch 32', 'Batch 8\nAccum 4'], rotation=0)
axes[1, 1].set_ylabel('Total Time (seconds)')
axes[1, 1].set_title('Training Time Comparison')
axes[1, 1].grid(axis='y', alpha=0.3)

# Add speedup annotation
speedup = times[1] / times[0]
axes[1, 1].text(1, times[1] + 0.5, f'{speedup:.2f}x\nslower', 
               ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Statistical comparison
print("\n" + "=" * 60)
print("STATISTICAL COMPARISON")
print("=" * 60)

standard = all_results["Standard (Batch 32)"]
accumulated = all_results["Accumulated (Batch 8, Accum 4)"]

print(f"\nFinal Validation Accuracy:")
print(f"  Standard:    {standard['val_accs'][-1]:.2f}%")
print(f"  Accumulated: {accumulated['val_accs'][-1]:.2f}%")
print(f"  Difference:  {abs(standard['val_accs'][-1] - accumulated['val_accs'][-1]):.2f}%")

print(f"\nAverage Gradient Norm:")
print(f"  Standard:    {np.mean(standard['grad_norms']):.4f}")
print(f"  Accumulated: {np.mean(accumulated['grad_norms']):.4f}")
print(f"  Difference:  {abs(np.mean(standard['grad_norms']) - np.mean(accumulated['grad_norms'])):.4f}")

print(f"\nTraining Time:")
print(f"  Standard:    {standard['time']:.2f}s")
print(f"  Accumulated: {accumulated['time']:.2f}s")
print(f"  Slowdown:    {accumulated['time'] / standard['time']:.2f}x")

print("\n" + "=" * 60)
print("CONCLUSION")
print("=" * 60)
print("✓ Training curves are virtually identical")
print("✓ Final accuracy is the same")
print("✓ Gradient norms match")
print("✓ Time trade-off: ~4x slower (as expected)")
print("\nGradient accumulation works correctly!")
print("=" * 60)

## Part 6: Common Pitfalls and Best Practices

### Pitfall 1: BatchNorm with Gradient Accumulation

**Problem**: BatchNorm computes statistics over the batch
```python
# With accumulation, each micro-batch has different statistics
# This causes training instability

Batch 32:        Uses 32 samples for mean/var
Batch 8 (x4):    Uses only 8 samples per forward pass
                 → Noisier statistics
                 → Different normalization each time
```

**Solution**: Use LayerNorm or RMSNorm instead
```python
# AVOID with gradient accumulation
self.norm = nn.BatchNorm1d(hidden_dim)

# USE instead
self.norm = nn.LayerNorm(hidden_dim)  # Transformers use this
```

### Pitfall 2: Incorrect Gradient Scaling

**Problem**: Forgetting to scale loss or gradients
```python
# WRONG: Gradients will be accumulation_steps × too large
for step in range(accumulation_steps):
    loss = criterion(model(x), y)
    loss.backward()  # ❌ No scaling!

# CORRECT: Scale loss
for step in range(accumulation_steps):
    loss = criterion(model(x), y)
    (loss / accumulation_steps).backward()  # ✓
```

### Pitfall 3: Learning Rate Not Adjusted

**Problem**: Changing effective batch size without adjusting LR
```python
# You tune with batch=32, LR=1e-4
# Then switch to batch=8, accum=4 (still effective=32)
# LR should stay 1e-4!

# But if you change effective batch:
# Old: batch=32, accum=1 → effective=32, LR=1e-4
# New: batch=8, accum=2 → effective=16, LR=~7e-5 (adjust!)
```

### Pitfall 4: Gradient Clipping at Wrong Time

**Problem**: Clipping before accumulation is complete
```python
# WRONG: Clips each micro-batch separately
for step in range(accumulation_steps):
    loss.backward()
    torch.nn.utils.clip_grad_norm_()  # ❌ Too early!
    if (step + 1) % accumulation_steps == 0:
        optimizer.step()

# CORRECT: Clip after accumulation
for step in range(accumulation_steps):
    loss.backward()
    if (step + 1) % accumulation_steps == 0:
        torch.nn.utils.clip_grad_norm_()  # ✓ After accumulation
        optimizer.step()
```

### Pitfall 5: Forgetting to Handle Last Batch

**Problem**: Last batch might not be full accumulation size
```python
# If total steps = 37, accumulation_steps = 4:
# Steps 0-3, 4-7, 8-11, ..., 32-35, 36 ← Only 1 step!

# CORRECT: Handle last batch
for step, (inputs, targets) in enumerate(dataloader):
    loss.backward()
    
    # Update if: accumulation complete OR last batch
    if (step + 1) % accumulation_steps == 0 or (step + 1) == len(dataloader):
        optimizer.step()
        optimizer.zero_grad()
```

### Best Practices:

**1. Use with mixed precision**:
```python
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

for step in range(accumulation_steps):
    with autocast():
        loss = model(x)
    scaler.scale(loss / accumulation_steps).backward()
    
    if (step + 1) % accumulation_steps == 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
```

**2. Log effective batch size**:
```python
print(f"Batch size: {batch_size}")
print(f"Accumulation steps: {accumulation_steps}")
print(f"Effective batch size: {batch_size * accumulation_steps}")
```

**3. Test equivalence**:
```python
# Train with both settings, verify identical results
# Standard: batch=32, accum=1
# Accumulated: batch=8, accum=4
# Should get same loss/accuracy!
```

In [ ]:
# Demonstrate pitfalls
print("=" * 60)
print("COMMON PITFALLS DEMONSTRATION")
print("=" * 60)

# Pitfall 1: No gradient scaling
print("\nPitfall 1: Forgetting to scale loss")
print("-" * 60)

model_correct = SimpleClassifier().to(device)
model_wrong = SimpleClassifier().to(device)

# Copy weights to make identical
model_wrong.load_state_dict(model_correct.state_dict())

optimizer_correct = optim.Adam(model_correct.parameters(), lr=0.001)
optimizer_wrong = optim.Adam(model_wrong.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Create small batch
torch.manual_seed(42)
X = torch.randn(16, 20).to(device)
y = torch.randint(0, 2, (16,)).to(device)

accumulation_steps = 4
batch_size = 4

# CORRECT: Scale loss
optimizer_correct.zero_grad()
for i in range(accumulation_steps):
    X_batch = X[i*batch_size:(i+1)*batch_size]
    y_batch = y[i*batch_size:(i+1)*batch_size]
    loss = criterion(model_correct(X_batch), y_batch)
    (loss / accumulation_steps).backward()  # ✓ SCALED

grad_norm_correct = torch.cat([p.grad.flatten() for p in model_correct.parameters()]).norm().item()

# WRONG: No scaling
optimizer_wrong.zero_grad()
for i in range(accumulation_steps):
    X_batch = X[i*batch_size:(i+1)*batch_size]
    y_batch = y[i*batch_size:(i+1)*batch_size]
    loss = criterion(model_wrong(X_batch), y_batch)
    loss.backward()  # ❌ NOT SCALED

grad_norm_wrong = torch.cat([p.grad.flatten() for p in model_wrong.parameters()]).norm().item()

print(f"Correct gradient norm (with scaling): {grad_norm_correct:.4f}")
print(f"Wrong gradient norm (no scaling):     {grad_norm_wrong:.4f}")
print(f"Ratio: {grad_norm_wrong / grad_norm_correct:.2f}x (should be ~{accumulation_steps})")
print("\n⚠️  Without scaling, gradients are too large!")

# Pitfall 2: Learning rate not adjusted for effective batch size
print("\n" + "="*60)
print("Pitfall 2: Learning rate and effective batch size")
print("-" * 60)

print("\nRule: Keep LR same if effective batch size is same")
print("\nExample 1: Same effective batch")
print("  Config A: batch=32, accum=1, LR=1e-4 → effective=32")
print("  Config B: batch=8,  accum=4, LR=1e-4 → effective=32")
print("  ✓ Same LR because same effective batch size")

print("\nExample 2: Different effective batch")
print("  Config A: batch=32, accum=1, LR=1e-4 → effective=32")
print("  Config B: batch=8,  accum=2, LR=?    → effective=16")
print("  ⚠️  Should reduce LR! Suggested: 7e-5 (sqrt scaling)")

def suggest_lr(base_lr, base_batch, new_batch, scaling='sqrt'):
    if scaling == 'linear':
        return base_lr * (new_batch / base_batch)
    elif scaling == 'sqrt':
        return base_lr * np.sqrt(new_batch / base_batch)
    else:
        return base_lr

print("\nLearning Rate Scaling Calculator:")
print(f"  Base: batch=32, LR=1e-4")
for new_batch in [8, 16, 64, 128]:
    lr_linear = suggest_lr(1e-4, 32, new_batch, 'linear')
    lr_sqrt = suggest_lr(1e-4, 32, new_batch, 'sqrt')
    print(f"  New batch={new_batch:3d}: LR={lr_sqrt:.1e} (sqrt) or {lr_linear:.1e} (linear)")

print("\n" + "="*60)
print("KEY TAKEAWAYS")
print("="*60)
print("\n1. ALWAYS scale loss by 1/accumulation_steps")
print("2. Match LR to EFFECTIVE batch size, not physical batch size")
print("3. Clip gradients AFTER accumulation, not during")
print("4. Handle last batch correctly (may be incomplete)")
print("5. Use LayerNorm, not BatchNorm")
print("="*60)

## 📊 Summary

### What We Learned:

**1. Mathematical Foundation**:
- Gradients are additive
- Accumulation is mathematically equivalent to larger batch
- Key: `∇_total = (1/n) Σ ∇_i` regardless of how you compute it

**2. Memory vs Time Trade-off**:
```
Memory savings: ~Linear with accumulation_steps
Time cost:      ~Linear with accumulation_steps
```

**3. Effective Batch Size**:
- Most important metric: `batch_size × accumulation_steps`
- Affects learning rate, training stability, gradient noise
- Rule: Match LR to effective batch size

**4. Implementation Details**:
```python
# Core pattern
optimizer.zero_grad()
for step in range(accumulation_steps):
    loss = criterion(model(x), y)
    (loss / accumulation_steps).backward()  # Scale!
    
    if (step + 1) % accumulation_steps == 0:
        torch.nn.utils.clip_grad_norm_()  # Optional
        optimizer.step()
        optimizer.zero_grad()
```

**5. Common Pitfalls**:
- ❌ Forgetting to scale loss/gradients
- ❌ Using BatchNorm (use LayerNorm)
- ❌ Not adjusting LR for effective batch size
- ❌ Clipping gradients at wrong time
- ❌ Not handling last incomplete batch

### When to Use Gradient Accumulation:

**✓ Use when**:
- Training large models (7B+) on limited GPUs
- Want large effective batch size for stability
- Memory constrained but time is available
- Fine-tuning with long sequences

**✗ Don't use when**:
- You have enough GPU memory (no point)
- Time is critical (training deadline)
- Using BatchNorm (normalization issues)
- Batch size already small (below 8-16)

### Practical Guidelines:

**For Transformer Fine-Tuning**:
```
Target effective batch sizes:
- Classification:  16-32
- Generation:      32-64
- Instruction tuning: 64-128

If GPU memory limited:
- Start with batch_size = 1 or 2
- Set accumulation_steps to reach target
- Example: batch=2, accum=32 → effective=64
```

**Memory Calculation**:
```python
# Estimate memory per sample
memory_per_sample = (
    seq_len * hidden_dim * num_layers * 
    4 bytes * 2  # Float32, forward+backward
)

# Maximum batch size
max_batch = available_memory // memory_per_sample

# If max_batch < target_effective_batch:
accumulation_steps = target_effective_batch // max_batch
```

### Real-World Examples:

**Llama-7B on 24GB GPU**:
```
Without accumulation: Batch size 1-2 (unstable)
With accumulation:    Batch 1, accum 32 → effective 32 (stable)
Trade-off:            32x slower, but works!
```

**GPT-2 (1.5B) on 16GB GPU**:
```
Without accumulation: Batch size 4-8
With accumulation:    Batch 4, accum 4 → effective 16
Trade-off:            4x slower, better convergence
```

### Integration with Other Techniques:

**Combined approaches**:
```
1. Gradient Accumulation + Mixed Precision (FP16/BF16)
   → 2x memory reduction from FP16
   → Another Nx from accumulation

2. Gradient Accumulation + Gradient Checkpointing
   → Checkpoint reduces activation memory
   → Accumulation reduces batch memory
   → Can train very large models!

3. Gradient Accumulation + LoRA
   → LoRA reduces trainable parameters
   → Accumulation allows larger effective batch
   → Best of both worlds
```

### Performance Tips:

**1. Find optimal accumulation steps**:
```python
# Don't accumulate more than needed
# Too much accumulation = unnecessarily slow

# Target: Largest batch_size that fits in memory
# Then use minimal accumulation to reach effective batch
```

**2. Use mixed precision**:
```python
# Accumulation + FP16 = huge memory savings
from torch.cuda.amp import autocast, GradScaler
# See Notebook 20 for details
```

**3. Monitor gradient norms**:
```python
# Verify scaling is correct
# Gradient norms should be similar with/without accumulation
```

### Next Notebook:

**17_regularization_techniques.ipynb**

We'll explore:
- Dropout strategies for transformers
- Weight decay (AdamW vs Adam)
- Early stopping strategies
- Gradient clipping (works with accumulation!)
- Learning rate warmup
- Comparing effects on validation loss

---

**Historical Note**: Gradient accumulation was a key enabler for democratizing large model training. Before this technique was widely adopted (2018-2019), only large research labs with massive GPU clusters could train transformers. Now, researchers can train billion-parameter models on consumer GPUs by trading time for memory. This technique, combined with LoRA and quantization, has made fine-tuning accessible to everyone.